**Shortcut Distillation Column: Benzene-Toluene (Fenske-Underwood-Gilliland)**

In this tutorial we size a benzene-toluene distillation column with the shortcut (Fenske-Underwood-Gilliland) method, and validate the result against DWSIM's Shortcut Column (Peng-Robinson property package).

IDAES has no shortcut column unit model, so `shortcut_column.py` builds one as a **custom IDAES unit model**: a `UnitModelBlockData` subclass with its own state blocks, ports, variables and constraints, which can be connected in a flowsheet like any other unit.

### Feed and specifications:

| | |
|---|---|
| Feed | 1 kg/s, 50/50 mol benzene/toluene, 300 K (subcooled liquid), 1.01325 bar |
| Keys | light key benzene, heavy key toluene |
| Product specifications | 1 mol % toluene in the distillate, 1 mol % benzene in the bottoms |
| Reflux ratio | 1.5 |
| Condenser | total, 1.01325 bar; reboiler 1.01325 bar; stage height 0.5 m |

### Shortcut method:

* **Fenske**, minimum stages at total reflux:
$$N_{min} = \frac{\ln\left[\left(\frac{x_{LK}}{x_{HK}}\right)_D \left(\frac{x_{HK}}{x_{LK}}\right)_B\right]}{\ln \alpha_{LK}}$$
* **Underwood**, minimum reflux, with the root $\theta$ between the key volatilities:
$$\sum_j \frac{\alpha_j z_j}{\alpha_j - \theta} = 1 - q \qquad R_{min} + 1 = \sum_j \frac{\alpha_j x_{j,D}}{\alpha_j - \theta}$$
* **Gilliland** (Eduljee correlation), stages at the actual reflux:
$$\frac{N - N_{min}}{N + 1} = 0.75\left[1 - \left(\frac{R - R_{min}}{R + 1}\right)^{0.5668}\right]$$
* **Kirkbride**, feed stage location:
$$\frac{N_R}{N_S} = \left[\frac{z_{HK}}{z_{LK}} \left(\frac{x_{LK,B}}{x_{HK,D}}\right)^2 \frac{B}{D}\right]^{0.206}$$

The feed quality $q$ (1.29 for this subcooled feed) comes from the feed, saturated liquid and saturated vapor enthalpies. The condenser duty is $V\,(h_V - h_D)$ for the vapor leaving the top stage, and the reboiler duty follows from the overall energy balance.

---

> **Relative volatility at the feed conditions**
>
> DWSIM's shortcut column evaluates the relative volatility from the K-values at the **feed** conditions (300 K here), giving α ≈ 3.3. Inside the column, between 353 K and 383 K, benzene-toluene has α ≈ 2.4. The custom unit reproduces DWSIM's choice, using single phase Peng-Robinson states (`liquid_configuration` and `vapor_configuration` in `benzene_toluene.py`) to calculate $K_j = \phi_j^L / \phi_j^V$ at the feed. With a subcooled feed this underestimates the stages needed; the rigorous column checks the design.

In [ ]:
import idaes

print("IDAES version: ", idaes.__version__)

## Setting up and solving the flowsheet

The flowsheet is Feed → ShortcutColumn → distillate and bottoms Products, built in `shortcut_column.py`.

In [ ]:
from shortcut_column import setup_shortcut_model, report_shortcut_properties

# Create and solve the model
model, results = setup_shortcut_model(
    flow_mass=1.0,  # kg/s
    mole_frac={"benzene": 0.5, "toluene": 0.5},
    temperature=300,  # K
    pressure=101325,  # Pa
    reflux_ratio=1.5,
    mole_frac_lk_bottoms=0.01,
    mole_frac_hk_distillate=0.01,
    condenser_pressure=101325,  # Pa
    reboiler_pressure=101325,  # Pa
    stage_height=0.5,  # m
)
print(results.solver.termination_condition)

In [ ]:
from pyomo.environ import value

c = model.fs.column
print("Relative volatility (feed):  {0:.4f}".format(value(c.alpha["benzene"])))
print("Feed quality q:              {0:.4f}".format(value(c.q)))
print("Minimum stages (Fenske):     {0:.3f}".format(value(c.min_stages)))
print("Minimum reflux (Underwood):  {0:.4f}".format(value(c.min_reflux_ratio)))
print("Stages (Gilliland):          {0:.3f}".format(value(c.stages)))
print("Feed stage (Kirkbride):      {0:.2f} stages above the feed, {1:.2f} below".format(
    value(c.rectifying_stages), value(c.stripping_stages)))
print("Condenser / reboiler duty:   {0:.1f} / {1:.1f} kW".format(-value(c.condenser_duty) / 1000, value(c.reboiler_duty) / 1000))

In [ ]:
# Visualize (if needed)
model.fs.visualize("Shortcut Column Model")

## Validation against DWSIM

The DWSIM results are read directly from the solved `.dwxmz` file and compared with IDAES. The small differences in the stage numbers come from the relative volatility (0.3 %), which depends on the Peng-Robinson fugacities and the ChemSep data in each tool.

In [ ]:
from shortcut_column import read_dwsim_results, compare_with_dwsim

dwsim = read_dwsim_results("DWSim Shortcut Column Model.dwxmz")
compare_with_dwsim(model, dwsim)